In [1]:
# import libarries, define geometry and generate mesh
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.webgui import Draw as DrawGeo
shape = Rectangle(2,0.41).Circle(0.2,0.2,0.05).Reverse().Face()
shape.edges.name="cyl"
shape.edges.Min(X).name="inlet"
shape.edges.Max(X).name="outlet"
shape.edges.Min(Y).name="wall"
shape.edges.Max(Y).name="wall"
mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=0.07)).Curve(3)

In [2]:
DrawGeo (shape)
Draw (mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [3]:
# viscosity
nu = 0.001

In [4]:
k = 3  # Taylor-Hood discretization of degree k
V = VectorH1(mesh,order=k, dirichlet="wall|cyl|inlet")
Q = H1(mesh,order=k-1)
X = V*Q

In [5]:
gfu = GridFunction(X)
velocity, pressure = gfu.components
print(gfu.components)
# parabolic inflow at bc=1:
uin = CoefficientFunction((1.5*4*y*(0.41-y)/(0.41*0.41),0))
velocity.Set(uin, definedon=mesh.Boundaries("inlet"))

Draw(velocity,mesh,"u")
Draw(pressure,mesh,"p")

(<ngsolve.comp.ComponentGridFunction object at 0x7c37705206b0>, <ngsolve.comp.ComponentGridFunction object at 0x7c3770520650>)


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [6]:
(u,p), (v,q) = X.TnT()

a = BilinearForm(X)
stokes = (nu*InnerProduct(grad(u),grad(v))-div(u)*q-div(v)*p)*dx
a += stokes
a.Assemble()

f = LinearForm(X)   
f.Assemble()

res = f.vec - a.mat*gfu.vec
inv_stokes = a.mat.Inverse(X.FreeDofs())
gfu.vec.data += inv_stokes * res

In [7]:
sceneu = Draw(velocity,mesh,"u")
scenep = Draw(pressure,mesh,"p")

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

In [36]:
# IMEX

dt = 0.001
# matrix for implicit part of IMEX(1) scheme:
mstar = BilinearForm(X)
mstar += InnerProduct(u,v)*dx + dt*stokes
mstar.Assemble()
inv = mstar.mat.Inverse(X.FreeDofs())

In [37]:
conv = BilinearForm(X, nonassemble = True)
conv += (Grad(u) * u) * v * dx

In [38]:
tend = 0
gfut = GridFunction(gfu.space,multidim=0)

In [39]:
# implicit Euler/explicit Euler splitting method:
tend += 1
gfut.AddMultiDimComponent(gfu.vec)
t = 0; cnt = 0
while t < tend-0.5*dt:
    print ("\rt=", t, end="")

    conv.Assemble()
    res = (a.mat + conv.mat)* gfu.vec
    gfu.vec.data -= dt * inv * res    

    t = t + dt; cnt += 1
    sceneu.Redraw()
    scenep.Redraw()
    if cnt % 50 == 0:
        gfut.AddMultiDimComponent(gfu.vec)


t= 0craete bilinearformapplication
t= 0.999000000000000876

In [40]:
Draw (gfut.components[0], mesh, interpolate_multidim=True, animate=True,
      min=0, max=1.9, autoscale=False, vectors = True)
Draw (gfut.components[1], mesh, interpolate_multidim=True, animate=True,
      min=-0.5, max=1, autoscale=False)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene